In [ ]:
# ???????????
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer

from sklearn.ensemble import BaggingRegressor, BaggingClassifier, RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor, GradientBoostingClassifier
from xgboost import XGBRegressor, XGBClassifier

from sklearn.metrics import mean_squared_error, f1_score

In [ ]:
# ?????????? ?????????? seed
my_code = "Petrov"
seed_limit = 2 ** 32
my_seed = int.from_bytes(my_code.encode(), "little") % seed_limit

In [ ]:
# ?????? ?????? ?? ?????
example_data = pd.read_csv("datasets/Fish.csv")

In [ ]:
example_data.head()

In [ ]:
# ????????? ?????? ????????????? ? ???????? ???????
val_test_size = round(0.2*len(example_data))
print(val_test_size)

In [ ]:
# ???????? ?????????, ????????????? ? ???????? ???????
random_state = my_seed
train_val, test = train_test_split(example_data, test_size=val_test_size, random_state=random_state)
train, val = train_test_split(train_val, test_size=val_test_size, random_state=random_state)
print(len(train), len(val), len(test))

In [ ]:
# ???????? ? ???????? ???????? ??????????? ? ??????? [0,1].
# ??? ????????? ???????????? ?????????? ?????? ????????? ???????.
num_columns = ['Weight', 'Length1', 'Length2', 'Length3', 'Height', 'Width']

ct = ColumnTransformer(transformers=[('numerical', MinMaxScaler(), num_columns)], remainder='passthrough')
ct.fit(train)

In [ ]:
# ??????????? ????????, ??? ?????? ???????? ? DataFrame
sc_train = pd.DataFrame(ct.transform(train))
sc_test = pd.DataFrame(ct.transform(test))
sc_val = pd.DataFrame(ct.transform(val))

In [ ]:
# ????????????? ???????? ????????
column_names = num_columns + ['Species']
sc_train.columns = column_names
sc_test.columns = column_names
sc_val.columns = column_names

In [ ]:
# ???? ?????? ???? ??????, ??? ????? ??? xgboost
types = {
    'Weight' : 'float64',
    'Length1' : 'float64',
    'Length2' : 'float64',
    'Length3' : 'float64',
    'Height' : 'float64',
    'Width' : 'float64',
    'Species' : 'category'
}
sc_train = sc_train.astype(types)
sc_test = sc_test.astype(types)
sc_val = sc_val.astype(types)

In [ ]:
# ??????? ?1 - ?????? ????????? ????? ????????? ??????? ? ?????? ?????????
# https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingRegressor.html
# https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html
# https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html
# https://xgboost.readthedocs.io/en/latest/python/python_api.html#module-xgboost.sklearn

In [ ]:
# ???????? 4 ???????? ??????????, ??? ?? ??? ????? ????????????, ???? - ????????? ??????????
n = 4
labels = random.sample(num_columns, n)

y_label = labels[0]
x_labels = labels[1:]

print(x_labels)
print(y_label)

In [ ]:
# ??????? ??????????? ?????????
x_train = sc_train[x_labels]
x_test = sc_test[x_labels]
x_val = sc_val[x_labels]

y_train = sc_train[y_label]
y_test = sc_test[y_label]
y_val = sc_val[y_label]

In [ ]:
# ???????? 4 ?????? ? ?????????? ?????????? ????????? criterion: 'mse', 'friedman_mse', 'mae', 'poisson'.
# ?????? ???????????? ?????? ????????? ? ??????? ????????? ??????? ? ???????? ?? ?????????????.
# ??? ????????????? ?????????? ????????? splitter, max_depth, min_samples_split, min_samples_leaf
# ???????, ????? ?????? ?????? ?????? ????? ??????.
score_list = []
r_model_list = [BaggingRegressor(), RandomForestRegressor(), GradientBoostingRegressor(), XGBRegressor()]
for i in range(len(r_model_list)):
    r_model_list[i].fit(x_train, y_train)
    score_list.append(r_model_list[i].score(x_val, y_val))
    print(score_list[i])

In [ ]:
best_model_r = r_model_list[score_list.index(max(score_list))]
print(best_model_r)

In [ ]:
test_pred = best_model_r.predict(x_test)
print(mean_squared_error(y_test, test_pred), sep='\n')

In [ ]:
# ??????? ?2 - ?????? ????????? ????? ????????? ? ?????? ?????????????
# https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html
# https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
# https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html
# https://xgboost.readthedocs.io/en/latest/python/python_api.html#module-xgboost.sklearn

In [ ]:
# ???????? 2 ???????? ??????????, ??????? ????? ??????????? ????????? ?????? ??????
# ????? ?????? ?????? 'Species'
n = 2
x_labels = random.sample(num_columns, n)
y_label = 'Species'

print(x_labels)
print(y_label)

In [ ]:
# ??????? ??????????? ?????????
x_train = sc_train[x_labels]
x_test = sc_test[x_labels]
x_val = sc_val[x_labels]

y_train = sc_train[y_label]
y_test = sc_test[y_label]
y_val = sc_val[y_label]

In [ ]:
x_train

In [ ]:
# ???????? 4 ?????? ? ?????????? ?????????? ????????? criterion : 'gini', 'entropy' ? splitter : 'best', 'random'.
# ?????? ???????????? ?????? ????????????? ? ??????? ????????? ??????? ? ???????? ?? ?????????????.
# ??? ????????????? ?????????? ????????? max_depth, min_samples_split, min_samples_leaf
# ???????, ????? ?????? ?????? ?????? ????? ??????.
score_list = []
c_model_list = [BaggingClassifier(), RandomForestClassifier(), GradientBoostingClassifier(), XGBClassifier()]
for i in range(len(c_model_list)):
    c_model_list[i].fit(x_train, y_train)
    score_list.append(c_model_list[i].score(x_val, y_val))
    print(score_list[i])

In [ ]:
best_model_c = c_model_list[score_list.index(max(score_list))]
print(best_model_c)

In [ ]:
test_pred = best_model_c.predict(x_test)
f1 = f1_score(y_test, test_pred, average='weighted')
print(f1, sep='\n')